In [ ]:
from pgl import pglStorage, pglSession, pglBehavior, pglDataMatrix

In [2]:
pglStorage.print()

=================================== Sessions ===================================
(pglStorage:printSessions) Database: pgl.pgl
(pglStorage:printSessions) Metadata snapshot: 2026-10-19 15:27:16.413898-07:00
  Deleted sessions excluded.

Digital Brain Memory pilot
  s480
    sessionID: 90
    sessionID: 91 -> 92

imageTask
  s0000
    sessionID: 75 -> 76

Things MEG pilot
  s0002
    sessionID: 73 -> 88 (filtered) -> 89 (fosters)
  s0003
    sessionID: 77
  s0004
    sessionID: 71



True

In [3]:
session = pglSession.loadSession(92)

(pglStorage:loadSession) Loaded derived checkpoint 92; backed fields will load on demand.


In [ ]:
session.get

In [4]:
# Use the pglBehavior class from your package.
behavior = pglBehavior.extractSession(session)

# Access the extracted xarray Dataset.
data = behavior.data

# Inspect its structure.
print(behavior)
print(data)

❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌
(pglBehavior:_addColumns) trialVariables: 1 field(s) marked 'unsupported'. variableEyePosition: At least one object has no supported analysis
adapter.
❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌
❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌
(pglBehavior:_addColumns) events: 1 field(s) marked 'unsupported'. eventKey: events.key, row 0: unsupported scientific value of type dict.
Expected a numeric, Boolean, or string scalar, or None.
❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌
pglDataMatrix(name='behavior', kind=Dataset, dimensions=[trial=260, event=9056, parameterCalibrationPointComponent=2], revision=unpublished)
<xarray.Dataset> Size: 4MB
Dimensions:                             (trial: 260, event: 9056,
                                         parameterCalibrationPointComponent: 2)
Coordinates:
  *

In [5]:
import numpy as np

# If needed after restarting/reloading:
# behavior = pglBehavior.extractSession(session)

trialData = behavior.getTrialData(taskName="memoryDescriptionTask")
trials = behavior.getBehaviorTable(taskName="memoryDescriptionTask")

assert trialData.sizes["trial"] == 80
assert len(trials) == 80
assert "event" not in trialData.dims
assert trials.index.tolist() == trialData.coords["trial"].values.tolist()
assert trials["taskSaveName"].eq("memoryDescriptionTask").all()
assert trials["responseCount"].tolist() == trialData["responseCount"].values.tolist()

# Confirm that xarray and pandas use identical response timing.
np.testing.assert_allclose(trials["responseTime"].to_numpy(dtype=float), trialData["responseTime"].values, equal_nan=True)

print(trials[["runName", "sourceTrialNum", "parameterMovieNum", "parameterDescription", "responseCount", "response", "responseType", "responseTime"]].head(12))
print("\nResponse-count distribution:")
print(trials["responseCount"].value_counts(dropna=False).sort_index())
print("\nTiming diagnostics:")
print(trials.attrs["responseTimingDiagnostics"])

# Exercise the alternative policy and optional numbered response columns.
lastTrials = behavior.getBehaviorTable(taskName="memoryDescriptionTask", responsePolicy="last")
expandedTrials = behavior.getBehaviorTable(taskName="memoryDescriptionTask", includeAllResponses=True)
assert lastTrials["responseCount"].equals(trials["responseCount"])
assert expandedTrials["responseCount"].equals(trials["responseCount"])

# All-task projection expands calibration vectors without multiplying rows.
allTrials = behavior.getBehaviorTable(taskSelection="all")
assert len(allTrials) == 260
assert "parameterCalibrationPointComponent0" in allTrials.columns
assert "parameterCalibrationPointComponent1" in allTrials.columns

# Editing pandas must not change the comprehensive source matrix.
firstTrialID = trials.index[0]
originalMovieNum = behavior.data["parameterMovieNum"].sel(trial=firstTrialID).item()
trials.loc[firstTrialID, "parameterMovieNum"] = -999
assert behavior.data["parameterMovieNum"].sel(trial=firstTrialID).item() == originalMovieNum
trials.loc[firstTrialID, "parameterMovieNum"] = originalMovieNum

# Repeated taskSaveName values must be rejected as ambiguous.
try:
    behavior.getBehaviorTable(taskName="acknowledgeMessage")
except ValueError as error:
    print("\nExpected ambiguous-selection error:", error)
else:
    raise AssertionError("Repeated acknowledgement tasks should be ambiguous.")

print("\nInitial trial projection and pandas smoke tests passed.")

      runName  sourceTrialNum  parameterMovieNum  \
trial                                              
18       run2               0                  0   
19       run2               1                  1   
20       run2               2                  2   
21       run2               3                  3   
22       run2               4                  4   
23       run2               5                  5   
24       run2               6                  6   
25       run2               7                  7   
26       run2               8                  8   
27       run2               9                  9   
28       run2              10                 10   
29       run2              11                 11   

                                    parameterDescription  responseCount  \
trial                                                                     
18                     a man holding a plstic container               1   
19               sp,epme smashing bottles fill

In [6]:
import numpy as np
import pandas as pd
import xarray as xr

# --------------------------------------------------------------
# Five synthetic trials:
#   0: no response
#   1: one response, with recorded response/type zero
#   2: two responses deliberately out of timestamp order
#   3: negative response latency
#   4: accepted response with missing timestamp
# --------------------------------------------------------------
testTrialRecords = [
    {"runIndex": 0, "taskIndex": 0, "sourceTrialNum": 0, "trialStartTime": 0.0, "trialStartEventIndex": 0},
    {"runIndex": 0, "taskIndex": 0, "sourceTrialNum": 1, "trialStartTime": 10.0, "trialStartEventIndex": 1},
    {"runIndex": 0, "taskIndex": 0, "sourceTrialNum": 2, "trialStartTime": 20.0, "trialStartEventIndex": 3},
    {"runIndex": 0, "taskIndex": 0, "sourceTrialNum": 3, "trialStartTime": 30.0, "trialStartEventIndex": 6},
    {"runIndex": 0, "taskIndex": 0, "sourceTrialNum": 4, "trialStartTime": 40.0, "trialStartEventIndex": 8},
]

testEventRecords = [
    {"eventType": "trial", "eventTime": 0.0, "eventTrialIndex": 0},
    {"eventType": "trial", "eventTime": 10.0, "eventTrialIndex": 1},
    {"eventType": "subjectResponse", "eventTime": 12.0, "eventTrialIndex": 1, "eventResponse": 0, "eventResponseType": 0},
    {"eventType": "trial", "eventTime": 20.0, "eventTrialIndex": 2},
    {"eventType": "subjectResponse", "eventTime": 25.0, "eventTrialIndex": 2, "eventResponse": 3, "eventResponseType": 8},
    {"eventType": "subjectResponse", "eventTime": 23.0, "eventTrialIndex": 2, "eventResponse": 4, "eventResponseType": 9},
    {"eventType": "trial", "eventTime": 30.0, "eventTrialIndex": 3},
    {"eventType": "subjectResponse", "eventTime": 29.5, "eventTrialIndex": 3, "eventResponse": 1, "eventResponseType": 1},
    {"eventType": "trial", "eventTime": 40.0, "eventTrialIndex": 4},
    {"eventType": "subjectResponse", "eventTime": None, "eventTrialIndex": 4, "eventResponse": 2, "eventResponseType": 2},
]

for testRecord in testTrialRecords:
    testRecord.update({"taskName": "Synthetic task", "taskSaveName": "syntheticTask"})

for testRecord in testEventRecords:
    testRecord["eventScope"] = "task"

testParameterRecords = [
    {"text": None, "flag": None, "number": None},
    {"text": "", "flag": False, "number": 0},
    {"text": "recorded", "flag": True, "number": 7},
    {"text": None, "flag": False, "number": -1},
    {"text": "final", "flag": None, "number": None},
]

testData = xr.Dataset(coords={"trial": np.arange(5, dtype=np.int64), "event": np.arange(10, dtype=np.int64)})

# Use the current encoding machinery to construct a projection test fixture.
# This tests projection/conversion, not extraction from recorder objects.
testTrialFields = {field: field for field in testTrialRecords[0]}
testEventFields = {field: field for field in ("eventType", "eventTime", "eventTrialIndex", "eventScope", "eventResponse", "eventResponseType")}
testParameterFields = {"text": "parameterText", "flag": "parameterFlag", "number": "parameterNumber"}

pglBehavior._addColumns(testData, "trial", testTrialRecords, testTrialFields, sourceGroup="trialContext")
pglBehavior._addColumns(testData, "event", testEventRecords, testEventFields, sourceGroup="events")
pglBehavior._addColumns(testData, "trial", testParameterRecords, testParameterFields, sourceGroup="params")

testData.attrs.update({
    "behaviorContract": pglBehavior.sourceContract,
    "runSources": [{"runIndex": 0, "runName": "syntheticRun"}],
    "taskSources": [{"runIndex": 0, "taskIndex": 0, "taskName": "Synthetic task", "taskSaveName": "syntheticTask"}],
})

testBehavior = pglDataMatrix(testData, name="behaviorProjectionTest")
testSourceBefore = testData.copy(deep=True)

# A single-task run should resolve automatically.
testFirst = testBehavior.getBehaviorTable(includeAllResponses=True)
testLast = testBehavior.getBehaviorTable(responsePolicy="last")
testTrialData = testBehavior.getTrialData(includeAllResponses=True)

# --------------------------------------------------------------
# Response counts and unanswered trials.
# --------------------------------------------------------------
assert len(testFirst) == 5
assert testFirst["responseCount"].tolist() == [0, 1, 2, 1, 1]

for testField in ("response", "responseType", "responseTime", "responseEventIndex"):
    assert pd.isna(testFirst.loc[0, testField]), testField

# Recorded zero is a response, not a missing value.
assert testFirst.loc[1, "response"] == 0
assert testFirst.loc[1, "responseType"] == 0
assert testFirst.loc[1, "responseTime"] == 2.0

# First/last follows recorded event order, not increasing timestamp.
assert testFirst.loc[2, "response"] == 3
assert testFirst.loc[2, "responseType"] == 8
assert testFirst.loc[2, "responseTime"] == 5.0
assert testFirst.loc[2, "responseEventIndex"] == 4
assert testLast.loc[2, "response"] == 4
assert testLast.loc[2, "responseType"] == 9
assert testLast.loc[2, "responseTime"] == 3.0
assert testLast.loc[2, "responseEventIndex"] == 5
assert testLast["responseCount"].equals(testFirst["responseCount"])

# Numbered responses preserve order and source event identity.
assert testFirst.loc[2, "response1"] == 3
assert testFirst.loc[2, "response2"] == 4
assert testFirst.loc[2, "responseTime1"] == 5.0
assert testFirst.loc[2, "responseTime2"] == 3.0
assert testFirst.loc[2, "responseEventIndex1"] == 4
assert testFirst.loc[2, "responseEventIndex2"] == 5
assert pd.isna(testFirst.loc[1, "response2"])

# Negative latency survives; missing timestamps do not erase responses.
assert testFirst.loc[3, "responseTime"] == -0.5
assert testFirst.loc[4, "response"] == 2
assert pd.isna(testFirst.loc[4, "responseTime"])

testDiagnostics = {(entry["trialIndex"], entry["eventIndex"], entry["status"]) for entry in testFirst.attrs["responseTimingDiagnostics"]}
assert testDiagnostics == {(3, 7, "negativeLatency"), (4, 9, "missingResponseTimestamp")}

# --------------------------------------------------------------
# Nullable pandas representations preserve meaningful fill-like values.
# --------------------------------------------------------------
assert str(testFirst["parameterText"].dtype) == "string"
assert str(testFirst["parameterFlag"].dtype) == "boolean"
assert str(testFirst["parameterNumber"].dtype) == "Int64"

assert pd.isna(testFirst.loc[0, "parameterText"])
assert pd.isna(testFirst.loc[0, "parameterFlag"])
assert pd.isna(testFirst.loc[0, "parameterNumber"])
assert testFirst.loc[1, "parameterText"] == ""
assert not bool(testFirst.loc[1, "parameterFlag"])
assert testFirst.loc[1, "parameterNumber"] == 0

# Check the xarray masks as well as the pandas interpretation.
assert not testTrialData["parameterTextPresent"].values[0]
assert testTrialData["parameterTextPresent"].values[1]
assert testTrialData["parameterText"].values[1] == ""

# Shared xarray/pandas timing and source independence.
np.testing.assert_allclose(testFirst["responseTime"].to_numpy(dtype=float), testTrialData["responseTime"].values, equal_nan=True)
assert testFirst.index.tolist() == testTrialData.coords["trial"].values.tolist()
xr.testing.assert_identical(testData, testSourceBefore)

testFirst.loc[1, "parameterNumber"] = 999
testTrialData["parameterNumber"].values[1] = 888
xr.testing.assert_identical(testData, testSourceBefore)

# Nonbehavioral matrices must be rejected.
try:
    pglDataMatrix(xr.Dataset(), name="notBehavior").getBehaviorTable()
except ValueError:
    pass
else:
    raise AssertionError("A matrix without the behavioral contract was accepted.")

print("PASS: response counts, first/last order, numbered responses, timing diagnostics, nullable values, shared projection, and source independence.")

NameError: name 'pglDataMatrix' is not defined

In [5]:
pglStorage.print()

NameError: name 'pglStorage' is not defined

In [ ]:
from pgl import pglStorage; pglStorage.initDatabase()

In [2]:
from pgl.pglSettings import pglSettings
from pgl.pglSession import pglSession

settings = pglSettings(refreshDisplays=False)
behaviorSessionID = 92
msession = pglSession.loadSession(sessionID=behaviorSessionID, settings=settings)
if msession is None: raise RuntimeError("Could not load the behavioral checkpoint.")

print("Checkpoint:", msession.sessionID)
print("Checkpoint name:", msession.checkpointName)
print("Behavioral runs:", len(msession.runs or []))

from pgl.pglSettings import pglSettings
from pgl.pglSession import pglSession

settings = pglSettings(refreshDisplays=False)

assert msession.sessionID is not None, "Save an acquisition/root checkpoint first."
parentID = msession.sessionID

# Read the parent's authoritative name before saving.
parentBefore = pglSession.loadSession(sessionID=parentID, settings=settings)
if parentBefore is None: raise RuntimeError("Could not load the parent checkpoint.")

parentName = parentBefore.checkpointName
workingDescription = msession.description
expectedDescription = workingDescription if workingDescription is not None and workingDescription.strip() else None

# WRITES: may create a derived checkpoint and permanent payload files.
savedID = msession.saveSession(checkpointName="memory", settings=settings)
if savedID is None: raise RuntimeError("Save did not return a checkpoint ID.")

assert msession.sessionID == savedID
assert msession.checkpointName == "memory"
assert msession.description == expectedDescription

restored = pglSession.loadSession(sessionID=savedID, settings=settings)
if restored is None: raise RuntimeError("Could not reload the saved checkpoint.")

assert restored.checkpointName == "memory"
assert restored.description == expectedDescription

# A changed name must never be implemented by renaming the parent.
if parentName != "memory":
    assert savedID != parentID

parentAfter = pglSession.loadSession(sessionID=parentID, settings=settings)
if parentAfter is None: raise RuntimeError("Could not reload the parent checkpoint.")
assert parentAfter.checkpointName == parentName

print(f"Checkpoint {savedID}: checkpointName={restored.checkpointName!r}")
print("Checkpoint-name save/load and parent-preservation checks passed.")

(pglStorage:loadSession) Loaded derived checkpoint 92; backed fields will load on demand.
Checkpoint: 92
Checkpoint name: memory
Behavioral runs: 5
(pglStorage:loadSession) Loaded derived checkpoint 92; backed fields will load on demand.
(pglStorage:saveSession) Saving session: connecting to PostgreSQL...
(pglStorage:saveSession) Saving session: resolving storage destination...
(pglStorage:saveSession) Saving session: resolving checkpoint owner...
(pglStorage:saveSession) Saving session: checking current contents and preparing checkpoint...
(pglStorage:_saveDerivedSession) Saving session: exporting current contents to temporary staging...
(pglStorage:_saveDerivedSession) Saving session: comparing contents with the parent checkpoint...
❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌
(pglStorage:_saveDerivedSession) Session contents and save metadata match checkpoint 92. No new checkpoint was created; returning the existing ID. Make
changes before calling

In [ ]:
import json
import posixpath

run = msession.runs[0]
filesystem = run.filesystem
taskPath = posixpath.join(run.fullDataPath, "task02_eyeTrackingCalibration")
parametersPath = posixpath.join(taskPath, "parameters")

print("Parameter directory:", parametersPath)
print("\nDirectory entries:")

for entryPath in sorted(filesystem.ls(parametersPath, detail=False)):
    entryKind = "directory" if filesystem.isdir(entryPath) else "file"
    print(f"  {entryKind}: {entryPath}")

rootSettingsPath = posixpath.join(parametersPath, "settings.json")

if filesystem.isfile(rootSettingsPath):
    with filesystem.open(rootSettingsPath, "rt", encoding="utf-8") as stream:
        rootSettings = json.load(stream)

    print("\nRoot parameter settings:")
    print(json.dumps(rootSettings, indent=2, ensure_ascii=False))
else:
    print("\nNo settings.json directly inside parameters/.")

In [ ]:
import posixpath
import traceback
from pgl.pglSerialize import pglSerialize

run = msession.runs[0]
taskPath = posixpath.join(run.fullDataPath, "task02_eyeTrackingCalibration")

for componentName in ("state", "data"):
    componentPath = posixpath.join(taskPath, f"{componentName}.json")
    print(f"\nRestoring: {componentPath}")

    with run.filesystem.open(componentPath, "rb") as stream:
        content = stream.read()

    try:
        component = pglSerialize.fromJSON(content, filename=componentPath)
        if component is None:
            print("Deserializer returned None; include any preceding warnings.")
        else:
            print(f"Successfully restored: {type(component).__name__}")
    except Exception:
        traceback.print_exc()

In [ ]:
from pgl import pglStorage
pglStorage.print(details=["dataset", "dataset_members"], includeDeleted=True)

In [ ]:
import dask.array as da
import numpy as np
import xarray as xr

from pgl import pglDataset
from pgl.pglData import pglDataMatrix, pglDataMatrixReference

behavior = xr.Dataset(
    data_vars={
        "reactionTime": ("trial", np.array([0.42, np.nan, 0.61], dtype=np.float64)),
        "correct": ("trial", np.array([True, False, True], dtype=bool)),
        "condition": ("trial", np.array(["left", "right", "left"], dtype=str)),
    },
    coords={"trial": np.arange(3, dtype=np.int64)},
    attrs={"description": "Synthetic dataset product integration test"},
)

behavior["reactionTime"].attrs["units"] = "s"
matrix = pglDataMatrix(behavior.chunk({"trial": 2}), name="behavior")
matrix.provenance = {"builderName": "syntheticTest", "sourceSessionIDs": []}

dataset = pglDataset(name="matrix-persistence-test")
dataset.checkpointName = "Initial product"
dataset.data["behavior"] = matrix

firstID = dataset.save()
assert firstID is not None, "Initial dataset save failed"

loaded = pglDataset.load(datasetID=firstID)
assert loaded is not None, "Dataset loading failed"
assert loaded._sessionCache == {}, "Loading populated the session cache"
assert isinstance(loaded.data["behavior"], pglDataMatrixReference), "Loading opened the product instead of retaining a reference"

firstRevisionID = loaded.data["behavior"].matrixRevisionID
loaded.print(verbose=True)
assert isinstance(loaded.data["behavior"], pglDataMatrixReference), "Printing opened the scientific product"

# Saving unopened references should reuse the registered payload.
loaded.checkpointName = "Reference reused"
secondID = loaded.save()
assert secondID is not None and secondID != firstID, "A new dataset checkpoint was not created"

second = pglDataset.load(datasetID=secondID)
assert second is not None, "Second checkpoint loading failed"
assert second.data["behavior"].matrixRevisionID == firstRevisionID, "Unopened reference did not reuse its revision"
print("PASS: unopened products load without opening Zarr and reuse their registered revision.")

# Explicit access opens and caches the mutable working matrix.
opened = second.getData("behavior")
assert isinstance(opened, pglDataMatrix), "Product did not open as a matrix"

try:
    assert second.data["behavior"] is opened, "Opened product was not cached"
    assert second.getData("behavior") is opened, "Repeated access returned a different matrix"

    for variableName in behavior.data_vars:
        assert isinstance(opened.data[variableName].data, da.Array), f"{variableName} is not lazy"

    actual = opened.data.compute().set_xindex("trial")
    xr.testing.assert_identical(behavior, actual)
    print("PASS: explicit access is cached, lazy-backed, and preserves scientific contents.")

    # Saving an exposed working matrix deliberately creates a new revision.
    second.checkpointName = "Working matrix republished"
    thirdID = second.save()
    assert thirdID is not None and thirdID not in (firstID, secondID), "Third checkpoint save failed"

    third = pglDataset.load(datasetID=thirdID)
    assert third is not None, "Third checkpoint loading failed"
    assert isinstance(third.data["behavior"], pglDataMatrixReference), "Third checkpoint did not load as a reference"
    assert third.data["behavior"].matrixRevisionID != firstRevisionID, "Mutable working matrix incorrectly reused its source revision"

    original = pglDataset.load(datasetID=firstID)
    assert original.data["behavior"].matrixRevisionID == firstRevisionID, "Original saved product reference changed"

    print("PASS: republishing a working matrix creates a new revision; the original checkpoint is unchanged.")
finally:
    opened.close()

print(f"Created dataset checkpoints: {firstID}, {secondID}, {thirdID}")
print("Two small matrix bundles were retained in configured storage. No sessions were deleted.")

In [ ]:
%pip install "zarr>=2.18.7,<3" "numcodecs>=0.15,<0.16" "dask[array]"